# 03 - Agregação semanal por município

A base principal `posto_semana.parquet` conserva uma linha por posto, semana e combustível, com preços, endereço, identificador e número de coletas. A média municipal é calculada em duas etapas: primeiro a média das coletas de cada posto na semana; depois a média simples dessas médias, atribuindo o mesmo peso a cada posto identificado. Os mínimos, máximos e desvios municipais descrevem a distribuição dos preços médios semanais dos postos, e `coletas` permanece como a soma das coletas que originaram essas médias.

Os registros sinalizados no Notebook 02 continuam incluídos na base principal. Para medir sua influência sem presumir que sejam erros, este notebook também calcula uma análise de sensibilidade que exclui os sinalizados, recalcula as médias por posto e município, compara os resultados semanais e gera um resumo por município e combustível. Essa visão alternativa é diagnóstica e não substitui nem altera a série principal.

In [ ]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / "Dados").exists() and (RAIZ.parent / "Dados").exists():
    RAIZ = RAIZ.parent
assert (RAIZ / "Dados").exists(), "Confira a pasta aberta no VS Code."

ENTRADA = RAIZ / "Dados/Limpo/precos_rmjp_limpos.parquet"
PASTA_ANALITICO = RAIZ / "Dados/Analitico"
PASTA_TABELAS = RAIZ / "tabelas"
PASTA_ANALITICO.mkdir(parents=True, exist_ok=True)
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(ENTRADA)

colunas_obrigatorias = {
    "semana", "estado", "municipio", "produto", "valor", "id_posto",
    "cnpj_revenda", "cnpj_valido", "outlier_sinalizado"
}
ausentes = colunas_obrigatorias.difference(df.columns)
if ausentes:
    raise ValueError(f"Execute primeiro 01_ingestao_rmjp e 02_limpeza. Colunas ausentes: {sorted(ausentes)}")

chaves = ["semana", "estado", "municipio", "produto"]
chaves_posto = chaves + ["id_posto"]
campos_posto = {
    "revenda": ("revenda", "first"),
    "bairro": ("bairro", "first"),
    "logradouro": ("logradouro", "first"),
    "numero": ("numero", "first"),
    "complemento": ("complemento", "first"),
    "cep": ("cep", "first"),
    "bandeira": ("bandeira", "first")
}
campos_posto = {nome: agregacao for nome, agregacao in campos_posto.items() if nome in df.columns}
dados_postos = df.loc[df["cnpj_valido"] & df["id_posto"].notna()].copy()


def agregar_posto_semana(base):
    resultado = (
        base.groupby(chaves_posto, as_index=False, dropna=False)
        .agg(
            cnpj_revenda=("cnpj_revenda", "first"),
            preco_medio=("valor", "mean"),
            preco_min=("valor", "min"),
            preco_max=("valor", "max"),
            desvio_coletas=("valor", "std"),
            coletas=("valor", "size"),
            **campos_posto
        )
    )
    resultado["amplitude"] = resultado["preco_max"] - resultado["preco_min"]
    return resultado


def agregar_municipio(posto_semanal):
    resultado = (
        posto_semanal.groupby(chaves, as_index=False, dropna=False)
        .agg(
            preco_medio=("preco_medio", "mean"),
            preco_min=("preco_medio", "min"),
            preco_max=("preco_medio", "max"),
            desvio_padrao=("preco_medio", "std"),
            coletas=("coletas", "sum"),
            postos_distintos=("id_posto", "nunique")
        )
    )
    resultado["amplitude"] = resultado["preco_max"] - resultado["preco_min"]
    resultado["cv_pct"] = 100 * resultado["desvio_padrao"] / resultado["preco_medio"]
    return resultado


posto_semana = agregar_posto_semana(dados_postos)
posto_semana.to_parquet(PASTA_ANALITICO / "posto_semana.parquet", index=False)

municipio_semana = agregar_municipio(posto_semana)
municipio_semana.to_parquet(PASTA_ANALITICO / "municipio_semana.parquet", index=False)

precos = municipio_semana.pivot(
    index=["semana", "estado", "municipio"],
    columns="produto",
    values="preco_medio"
).reset_index()
precos.columns.name = None
precos.to_parquet(PASTA_ANALITICO / "precos_municipio_semana.parquet", index=False)

# Sensibilidade: recalcula os mesmos agregados sem os preços sinalizados, sem alterar a base principal.
sem_sinalizacao = ~dados_postos["outlier_sinalizado"].fillna(False).astype(bool)
posto_semana_sem_outliers = agregar_posto_semana(dados_postos.loc[sem_sinalizacao])
municipio_semana_sem_outliers = agregar_municipio(posto_semana_sem_outliers)

com_outliers = municipio_semana[
    chaves + ["preco_medio", "postos_distintos", "coletas"]
].rename(columns={
    "preco_medio": "preco_medio_com_outliers",
    "postos_distintos": "postos_com_outliers",
    "coletas": "coletas_com_outliers"
})
sem_outliers = municipio_semana_sem_outliers[
    chaves + ["preco_medio", "postos_distintos", "coletas"]
].rename(columns={
    "preco_medio": "preco_medio_sem_outliers",
    "postos_distintos": "postos_sem_outliers",
    "coletas": "coletas_sem_outliers"
})
impacto_sensibilidade = com_outliers.merge(
    sem_outliers, on=chaves, how="outer", validate="one_to_one"
)
contagem_sinalizados = (
    df.loc[df["outlier_sinalizado"].fillna(False).astype(bool)]
    .groupby(chaves, as_index=False)
    .size()
    .rename(columns={"size": "registros_sinalizados"})
)
impacto_sensibilidade = impacto_sensibilidade.merge(
    contagem_sinalizados, on=chaves, how="left", validate="one_to_one"
)
impacto_sensibilidade["registros_sinalizados"] = (
    impacto_sensibilidade["registros_sinalizados"].fillna(0).astype("int64")
)
impacto_sensibilidade["diferenca_preco"] = (
    impacto_sensibilidade["preco_medio_sem_outliers"]
    - impacto_sensibilidade["preco_medio_com_outliers"]
)
impacto_sensibilidade["variacao_pct"] = 100 * (
    impacto_sensibilidade["diferenca_preco"]
    / impacto_sensibilidade["preco_medio_com_outliers"]
)
impacto_sensibilidade.to_parquet(
    PASTA_ANALITICO / "municipio_semana_sensibilidade_outliers.parquet",
    index=False
)

comparaveis = impacto_sensibilidade.dropna(
    subset=["preco_medio_com_outliers", "preco_medio_sem_outliers"]
)
resumo_sensibilidade = (
    comparaveis.groupby(["municipio", "produto"], as_index=False)
    .agg(
        preco_medio_com_outliers=("preco_medio_com_outliers", "mean"),
        preco_medio_sem_outliers=("preco_medio_sem_outliers", "mean"),
        semanas_comparaveis=("semana", "nunique"),
        registros_sinalizados=("registros_sinalizados", "sum"),
        postos_semana_com_outliers=("postos_com_outliers", "mean"),
        postos_semana_sem_outliers=("postos_sem_outliers", "mean")
    )
)
resumo_sensibilidade["diferenca_preco"] = (
    resumo_sensibilidade["preco_medio_sem_outliers"]
    - resumo_sensibilidade["preco_medio_com_outliers"]
)
resumo_sensibilidade["variacao_pct"] = 100 * (
    resumo_sensibilidade["diferenca_preco"]
    / resumo_sensibilidade["preco_medio_com_outliers"]
)
resumo_sensibilidade.to_csv(
    PASTA_TABELAS / "impacto_sensibilidade_outliers.csv",
    index=False,
    sep=";",
    decimal=","
)

print("Registros preservados na base limpa:", len(df))
print("Registros válidos com posto identificado:", len(dados_postos))
print("Registros sinalizados na base:", int(df["outlier_sinalizado"].fillna(False).sum()))
print("Linhas posto-semana:", len(posto_semana))
print("Postos identificados:", posto_semana["id_posto"].nunique())
print("Linhas município-semana-combustível:", len(municipio_semana))
print("Grupos semanais comparáveis na sensibilidade:", len(comparaveis))
print("Grupos sem resultado sem outliers:", int(impacto_sensibilidade["preco_medio_sem_outliers"].isna().sum()))
print("Período:", municipio_semana["semana"].min(), "até", municipio_semana["semana"].max())
display(municipio_semana.head())
display(resumo_sensibilidade)

Linhas posto-semana: 69285
Postos com CNPJ válido: 205
Linhas município-semana-produto: 5249
Período: 2011-01-03 00:00:00 até 2026-06-29 00:00:00


,semana,estado,municipio,id_posto,produto,cnpj_revenda,preco_medio,preco_min,preco_max,desvio_coletas,coletas,revenda,bairro,logradouro,numero,complemento,cep,bandeira,amplitude
0,2011-01-03,PB,BAYEUX,01420327000185,ETANOL,01420327000185,1.79,1.79,1.79,<NA>,1,MOTOGAS INDUSTRIA DE COMPRESSAO E COMERCIO DE ...,ESTRADA,LOC BR 230 KM 31,S/N,NaN,58309602,BRANCA,0.0
1,2011-01-03,PB,BAYEUX,01420327000185,GASOLINA,01420327000185,2.48,2.48,2.48,<NA>,1,MOTOGAS INDUSTRIA DE COMPRESSAO E COMERCIO DE ...,ESTRADA,LOC BR 230 KM 31,S/N,NaN,58309602,BRANCA,0.0
2,2011-01-03,PB,BAYEUX,01633732000181,ETANOL,01633732000181,1.79,1.79,1.79,<NA>,1,COMERCIAL DE COMBUSTIVEIS ENTRE RIOS LTDA,SESI,AVENIDA LIBERDADE,2212,NaN,58305000,BRANCA,0.0
3,2011-01-03,PB,BAYEUX,01633732000181,GASOLINA,01633732000181,2.28,2.28,2.28,<NA>,1,COMERCIAL DE COMBUSTIVEIS ENTRE RIOS LTDA,SESI,AVENIDA LIBERDADE,2212,NaN,58305000,BRANCA,0.0
4,2011-01-03,PB,BAYEUX,03450959000117,ETANOL,03450959000117,1.79,1.79,1.79,<NA>,1,POSTO ESTRELA DA LIBERDADE LTDA,SAO BENTO,"AVENIDA LIBERDADE ,",1232,1268,58309602,DISLUB,0.0


,semana,estado,municipio,produto,preco_medio,preco_min,preco_max,desvio_padrao,coletas,postos_distintos,amplitude,cv_pct
0,2011-01-03,PB,BAYEUX,ETANOL,1.822222,1.78,1.94,0.066854,9,9,0.16,3.668812
1,2011-01-03,PB,BAYEUX,GASOLINA,2.385556,2.28,2.56,0.120531,9,9,0.28,5.052543
2,2011-01-03,PB,CABEDELO,ETANOL,1.875556,1.76,1.96,0.069482,9,9,0.2,3.70462
3,2011-01-03,PB,CABEDELO,GASOLINA,2.48,2.29,2.59,0.095131,9,9,0.3,3.835947
4,2011-01-03,PB,JOAO PESSOA,ETANOL,1.805583,1.689,2.05,0.090558,48,48,0.361,5.015458
